# SpectraVault AI — Advanced Hybrid Medical Data Embedding
### Digital Image Processing Project | AI-Augmented Secure Steganography
**Tech Stack:** AES-256-GCM | LSB | DCT (Frequency Domain) | FFT | DenseNet-121



In [1]:
!pip install medmnist torch torchvision opencv-python matplotlib numpy pandas \
             scipy pycryptodome scikit-image torchxrayvision -q
print(" All libraries installed!")

 All libraries installed!


In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import base64
import json
import os
import csv
import sys
from pathlib import Path
from getpass import getpass
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Random import get_random_bytes

import torch
import torchvision.transforms as transforms
from medmnist import ChestMNIST
from scipy.fftpack import dct, idct
from skimage.metrics import structural_similarity as ssim_func
import pandas as pd

# ── Project root & spectravault package ─────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.ipynb").exists():
    PROJECT_ROOT = Path("/content") if Path("/content").exists() else PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT))

from spectravault.paths import get_output_dirs
from spectravault.pipeline import (
    run_lsb_pipeline,
    export_frontend_metrics,
    build_lsb_summary,
    evaluate_lsb_split,
    lsb_round_trip,
)
from spectravault.encryption import MediGuardEncryptor
from spectravault.lsb_stego import LSBStego
from spectravault.metrics import calculate_metrics, quality_label
from spectravault.metadata import get_patient_metadata, structure_for_embedding, pil_to_rgb_array, DISEASES

_dirs = get_output_dirs(PROJECT_ROOT)
OUTPUT_DIR  = _dirs["output"]
STEGO_DIR   = _dirs["stego"]
METRICS_DIR = _dirs["metrics"]
MODELS_DIR  = _dirs["models"]
FIGS_DIR    = _dirs["figures"]
LSB_DIR     = _dirs["lsb"]

print(" All imports successful!")
print(f" Outputs will be saved to: {OUTPUT_DIR}")
print(" LSB pipeline module: spectravault/ (run Step 11 for LSB-only evaluation)")


 All imports successful!
 Outputs will be saved to: C:\Users\HP\Desktop\SpectraVault-AI-1\mediguard_outputs
 LSB pipeline module: spectravault/ (run Step 11 for LSB-only evaluation)


## Step 1 — Configuration
Password is entered securely via `getpass` — never hardcoded.

In [3]:
from getpass import getpass
print(" MediGuard — Encryption Setup")
print("=" * 45)
ENCRYPTION_PASSWORD = getpass("Enter encryption password: ")
CONFIRM_PASSWORD    = getpass("Confirm password          : ")

if ENCRYPTION_PASSWORD != CONFIRM_PASSWORD:
    raise ValueError(" Passwords do not match. Re-run this cell.")

print(" Password accepted.\n")

CONFIG = {
    "image_size"      : 224,
    "lsb_bits"        : 1,
    "dct_block_size"  : 8,
    "dct_coefficient" : (4, 3),
    "dct_alpha"       : 8,
    "fft_freq_band"   : (50, 100),
    "fft_strength"    : 5.0,
    "ai_threshold"    : 0.30,   # confidence threshold for DenseNet predictions
    "eval_n_images"   : 20,     # images to evaluate in train/val/test comparison
    "dct_image_size" : 512,   # ← add this line
}

print(" CONFIG:")
for k, v in CONFIG.items():
    print(f"   {k:20s}: {v}")

 MediGuard — Encryption Setup


Enter encryption password:  ········
Confirm password          :  ········


 Password accepted.

 CONFIG:
   image_size          : 224
   lsb_bits            : 1
   dct_block_size      : 8
   dct_coefficient     : (4, 3)
   dct_alpha           : 8
   fft_freq_band       : (50, 100)
   fft_strength        : 5.0
   ai_threshold        : 0.3
   eval_n_images       : 20
   dct_image_size      : 512


## Step 2 — Load MedMNIST ChestMNIST Dataset
Real medical X-rays (112K images, 14 disease labels).  


In [5]:
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

print(" Downloading ChestMNIST …")
train_dataset = ChestMNIST(split='train', download=True, size=CONFIG["image_size"])
val_dataset   = ChestMNIST(split='val',   download=True, size=CONFIG["image_size"])
test_dataset  = ChestMNIST(split='test',  download=True, size=CONFIG["image_size"])

print(f" Dataset loaded!")
print(f"   Train : {len(train_dataset):>6,} images")
print(f"   Val   : {len(val_dataset):>6,} images")
print(f"   Test  : {len(test_dataset):>6,} images")
print(f"   Total : {len(train_dataset)+len(val_dataset)+len(test_dataset):>6,} images")

# Quick sanity check
sample_img, sample_label = train_dataset[0]
active_diseases = [DISEASES[i] for i, v in enumerate(sample_label) if v == 1]
print(f"\n Sample image[0] — real disease labels from dataset:")
print(f"   {active_diseases if active_diseases else ['No Finding']}")

 Dataset loaded!
   Train : 78,468 images
   Val   : 11,219 images
   Test  : 22,433 images
   Total : 112,120 images

 Sample image[0] — real disease labels from dataset:
   ['No Finding']


## Step 3 — AI Disease Prediction (DenseNet-121)
Uses **torchxrayvision** pretrained on NIH ChestX-ray14 (same domain).  


In [6]:
try:
    import torchxrayvision as xrv

    print(" Loading pretrained DenseNet-121 (NIH ChestX-ray14 weights) …")
    ai_model = xrv.models.DenseNet(weights="densenet121-res224-nih")
    ai_model.eval()

    XRV_DISEASES = [p for p in ai_model.pathologies if p]
    print(f" DenseNet-121 loaded  |  {len(XRV_DISEASES)} pathology outputs")

    def predict_diseases_ai(pil_img):
        """Run DenseNet-121 on a PIL image, return (detected_dict, full_confidence_dict)."""
        if pil_img.mode != 'L':
            pil_img = pil_img.convert('L')
        img_t = transforms.ToTensor()(pil_img.resize((224, 224)))
        img_t = img_t * 2048 - 1024          # torchxrayvision expects [-1024, 1024]
        img_t = img_t.unsqueeze(0)
        with torch.no_grad():
            probs = torch.sigmoid(ai_model(img_t)).squeeze().numpy()
        full_conf = {XRV_DISEASES[i]: round(float(probs[i]), 4) for i in range(len(XRV_DISEASES))}
        detected  = {k: v for k, v in full_conf.items() if v >= CONFIG["ai_threshold"]}
        return detected, full_conf

    AI_AVAILABLE = True
    # Quick test
    sample_det, sample_conf = predict_diseases_ai(train_dataset[0][0])
    print(f"\n DenseNet predictions on sample image (threshold={CONFIG['ai_threshold']}):")
    for disease, conf in sorted(sample_det.items(), key=lambda x: -x[1]):
        print(f"   {disease:<22} {conf:.3f}")
    if not sample_det:
        print("   (No findings above threshold — top-3 anyway:)")
        for d, c in sorted(sample_conf.items(), key=lambda x:-x[1])[:3]:
            print(f"   {d:<22} {c:.3f}")

except ImportError:
    print("  torchxrayvision not found — falling back to dataset labels only.")
    AI_AVAILABLE = False
    def predict_diseases_ai(pil_img):
        return {}, {}

 Loading pretrained DenseNet-121 (NIH ChestX-ray14 weights) …
 DenseNet-121 loaded  |  14 pathology outputs

 DenseNet predictions on sample image (threshold=0.3):
   Atelectasis            0.626
   Fibrosis               0.625
   Nodule                 0.624
   Pleural_Thickening     0.623
   Consolidation          0.623
   Hernia                 0.623
   Pneumonia              0.623
   Emphysema              0.623
   Infiltration           0.618
   Pneumothorax           0.608
   Mass                   0.588
   Edema                  0.587
   Cardiomegaly           0.572
   Effusion               0.562


## Step 4 — Patient Metadata Extraction (Honest)


In [7]:
# Clinical epidemiology-based demographic priors (from published literature)
DISEASE_EPI = {
    'Emphysema'          : {'age_μ': 62, 'age_σ': 8,  'male_p': 0.65},
    'Fibrosis'           : {'age_μ': 58, 'age_σ': 10, 'male_p': 0.55},
    'Cardiomegaly'       : {'age_μ': 60, 'age_σ': 12, 'male_p': 0.50},
    'Effusion'           : {'age_μ': 55, 'age_σ': 14, 'male_p': 0.52},
    'Pneumonia'          : {'age_μ': 45, 'age_σ': 20, 'male_p': 0.52},
    'Pneumothorax'       : {'age_μ': 35, 'age_σ': 15, 'male_p': 0.70},
    'Mass'               : {'age_μ': 57, 'age_σ': 12, 'male_p': 0.55},
    'Nodule'             : {'age_μ': 54, 'age_σ': 12, 'male_p': 0.53},
    'Atelectasis'        : {'age_μ': 50, 'age_σ': 16, 'male_p': 0.50},
    'Consolidation'      : {'age_μ': 48, 'age_σ': 18, 'male_p': 0.51},
    'Edema'              : {'age_μ': 58, 'age_σ': 13, 'male_p': 0.50},
    'Infiltration'       : {'age_μ': 47, 'age_σ': 17, 'male_p': 0.51},
    'Hernia'             : {'age_μ': 55, 'age_σ': 12, 'male_p': 0.65},
    'Pleural_Thickening' : {'age_μ': 52, 'age_σ': 14, 'male_p': 0.55},
}

def get_patient_metadata(index, dataset=train_dataset, use_ai=True):
    rng = np.random.default_rng(seed=index + 2026)  # reproducible per patient
    pil_img, label = dataset[index]

    # ── Real disease labels ───────────────────────────────────────────────────
    real_diseases = [DISEASES[i] for i, v in enumerate(label) if v == 1]

    # ── AI predictions ───────────────────────────────────────────────────────
    ai_detected, ai_conf = {}, {}
    if use_ai and AI_AVAILABLE:
        ai_detected, ai_conf = predict_diseases_ai(pil_img)

    # Prefer AI output if available; else fall back to real labels
    if ai_detected:
        final_diagnoses = list(ai_detected.keys())
        dx_source = 'DenseNet121_AI'
    elif real_diseases:
        final_diagnoses = real_diseases
        dx_source = 'MedMNIST_labels'
    else:
        final_diagnoses = ['No Finding']
        dx_source = 'MedMNIST_labels'

    # ── Simulated demographics (clearly labelled) ────────────────────────────
    primary = final_diagnoses[0] if final_diagnoses[0] != 'No Finding' else None
    epi = DISEASE_EPI.get(primary, {'age_μ': 50, 'age_σ': 15, 'male_p': 0.5}) if primary else {'age_μ': 50, 'age_σ': 15, 'male_p': 0.5}
    age    = int(np.clip(rng.normal(epi['age_μ'], epi['age_σ']), 18, 90))
    gender = 'M' if rng.random() < epi['male_p'] else 'F'
    view   = rng.choice(['PA', 'AP'], p=[0.65, 0.35])

    return {
        'patient_id'         : f'P{index:05d}',
        'image_index'        : index,
        'diagnoses_source'   : dx_source,
        'diagnoses'          : '|'.join(final_diagnoses),
        'ai_confidence'      : json.dumps({k: round(v,3) for k,v in ai_conf.items() if v >= 0.15}),
        'demographics_source': 'SIMULATED_epidemiology_based',
        'age'                : age,
        'gender'             : gender,
        'view_position'      : view,
    }

def structure_for_embedding(meta):
    """Compact string written into the stego image."""
    return (
        f"PID:{meta['patient_id']}|"
        f"AGE:{meta['age']}|"
        f"GEN:{meta['gender']}|"
        f"VIEW:{meta['view_position']}|"
        f"SRC:{meta['diagnoses_source']}|"
        f"DX:{meta['diagnoses']}"
    )

# Test
meta0 = get_patient_metadata(0)
print(" Patient 0 metadata:")
for k, v in meta0.items():
    print(f"   {k:<24}: {v}")
s0 = structure_for_embedding(meta0)
print(f"\n Embedding string ({len(s0)} chars):")
print(f"   {s0}")

 Patient 0 metadata:
   patient_id              : P00000
   image_index             : 0
   diagnoses_source        : DenseNet121_AI
   diagnoses               : Atelectasis|Consolidation|Infiltration|Pneumothorax|Edema|Emphysema|Fibrosis|Effusion|Pneumonia|Pleural_Thickening|Cardiomegaly|Nodule|Mass|Hernia
   ai_confidence           : {"Atelectasis": 0.626, "Consolidation": 0.623, "Infiltration": 0.618, "Pneumothorax": 0.608, "Edema": 0.587, "Emphysema": 0.623, "Fibrosis": 0.625, "Effusion": 0.562, "Pneumonia": 0.623, "Pleural_Thickening": 0.623, "Cardiomegaly": 0.572, "Nodule": 0.624, "Mass": 0.588, "Hernia": 0.623}
   demographics_source     : SIMULATED_epidemiology_based
   age                     : 37
   gender                  : F
   view_position           : PA

 Embedding string (200 chars):
   PID:P00000|AGE:37|GEN:F|VIEW:PA|SRC:DenseNet121_AI|DX:Atelectasis|Consolidation|Infiltration|Pneumothorax|Edema|Emphysema|Fibrosis|Effusion|Pneumonia|Pleural_Thickening|Cardiomegaly|Nodul

## Step 5 — AES-256-GCM Encryption

In [8]:
class MediGuardEncryptor:
    """AES-256-GCM with PBKDF2 key derivation. Password set at runtime (not hardcoded)."""

    def __init__(self, password: str):
        self._pw = password.encode()

    def encrypt(self, plaintext: str) -> str:
        salt  = get_random_bytes(16)
        nonce = get_random_bytes(16)
        key   = PBKDF2(self._pw, salt, dkLen=32, count=100_000)
        cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
        ct, tag = cipher.encrypt_and_digest(plaintext.encode())
        # Layout: salt(16) | nonce(16) | tag(16) | ciphertext
        return base64.b64encode(salt + nonce + tag + ct).decode()

    def decrypt(self, b64: str) -> str:
        raw   = base64.b64decode(b64)
        salt, nonce, tag, ct = raw[:16], raw[16:32], raw[32:48], raw[48:]
        key   = PBKDF2(self._pw, salt, dkLen=32, count=100_000)
        cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
        return cipher.decrypt_and_verify(ct, tag).decode()

encryptor = MediGuardEncryptor(ENCRYPTION_PASSWORD)

# ── Verify encrypt → decrypt round-trip ──────────────────────────────────────
_test_msg = structure_for_embedding(meta0)
_enc      = encryptor.encrypt(_test_msg)
_dec      = encryptor.decrypt(_enc)

print(" AES-256-GCM Round-Trip Test:")
print(f"   Original  : {_test_msg[:70]}…")
print(f"   Encrypted : {_enc[:70]}…")
print(f"   Decrypted : {_dec[:70]}…")
print(f"   Match   : {_dec == _test_msg}")
print(f"   Payload length after encryption: {len(_enc)} chars / {len(_enc)*8} bits")

 AES-256-GCM Round-Trip Test:
   Original  : PID:P00000|AGE:37|GEN:F|VIEW:PA|SRC:DenseNet121_AI|DX:Atelectasis|Cons…
   Encrypted : 4w+V/EHIEnSzTwVaMEQChQv6MvjQIf/TDLNQaj0Xw8x3i5pd5OM/bFoYFM3eWziUYgAyHf…
   Decrypted : PID:P00000|AGE:37|GEN:F|VIEW:PA|SRC:DenseNet121_AI|DX:Atelectasis|Cons…
   Match   : True
   Payload length after encryption: 332 chars / 2656 bits


## Step 6 — LSB Steganography (Spatial Domain)

In [9]:
class LSBStego:
    """Least-Significant-Bit spatial-domain steganography.
    Blind extraction — only the stego image is needed.
    Header: 32-bit message-length prefix.
    """
    def __init__(self, bits: int = 1):
        self.bits = bits
        self.mask = (1 << bits) - 1
        self.clear = 0xFF ^ self.mask

    def _text_to_bits(self, text):
        msg = ''.join(format(ord(c), '08b') for c in text)
        return format(len(msg), '032b') + msg

    def _bits_to_text(self, bits_str):
        n = int(bits_str[:32], 2)
        b = bits_str[32:32 + n]
        return ''.join(chr(int(b[i:i+8], 2)) for i in range(0, len(b), 8))

    def embed(self, image_array: np.ndarray, message: str) -> np.ndarray:
        img  = image_array.copy().astype(np.uint8)
        flat = img.flatten()
        full = self._text_to_bits(message)
        if len(full) > len(flat):
            raise ValueError(f"Message too large: {len(full)} bits > {len(flat)} capacity")
        for i, bit in enumerate(full):
            flat[i] = (flat[i] & self.clear) | int(bit)
        return flat.reshape(img.shape)

    def extract(self, stego_array: np.ndarray) -> str:
        flat = stego_array.flatten().astype(np.uint8)
        bits = ''.join(str(p & self.mask) for p in flat)
        return self._bits_to_text(bits)

# Quick test
lsb_stego = LSBStego(bits=CONFIG["lsb_bits"])
_img0 = np.array(train_dataset[0][0].resize((224,224)))
if _img0.ndim == 2: _img0 = np.stack([_img0]*3, axis=-1)
_enc0 = encryptor.encrypt(structure_for_embedding(meta0))
_s    = lsb_stego.embed(_img0.copy(), _enc0)
_out  = encryptor.decrypt(lsb_stego.extract(_s))
print(f" LSB round-trip: {_out == structure_for_embedding(meta0)}")
print(f"   Image capacity: {_img0.size} bits | Used: {len(_enc0)*8 + 32} bits")

 LSB round-trip: True
   Image capacity: 150528 bits | Used: 2688 bits


## Step 7 — DCT Steganography (Frequency Domain — BLIND)


**Method:** Quantization-parity embedding (JSteg-style)  
- Embed `0` → round coefficient to nearest *even* multiple of `alpha`  
- Embed `1` → round coefficient to nearest *odd* multiple of `alpha`  
- Extract → check parity of `⌊coefficient / alpha⌋`  
No original image needed for extraction

In [10]:
class DCTStegoBlind:
    """
    Blind DCT steganography — quantization-parity approach.
    Extraction does NOT need the original image.
    """
    def __init__(self, block_size=8, coefficient=(4, 3), alpha=8):
        self.bs   = block_size
        self.ci, self.cj = coefficient
        self.alpha = alpha

    # ── helpers ──────────────────────────────────────────────────────────────
    def _text_to_bits(self, text):
        msg = ''.join(format(ord(c), '08b') for c in text)
        return format(len(msg), '032b') + msg

    def _bits_to_text(self, bits):
        n = int(bits[:32], 2)
        b = bits[32:32 + n]
        return ''.join(chr(int(b[i:i+8], 2)) for i in range(0, len(b), 8))

    def _to_gray(self, arr):
        if arr.ndim == 3:
            return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float32)
        return arr.astype(np.float32)

    def _pad(self, gray):
        h, w = gray.shape
        ph = (-h) % self.bs
        pw = (-w) % self.bs
        return np.pad(gray, ((0, ph), (0, pw)), 'edge'), h, w

    # ── embed ─────────────────────────────────────────────────────────────────
    def embed(self, image_array: np.ndarray, message: str) -> np.ndarray:
        gray = self._to_gray(image_array)
        padded, h, w = self._pad(gray)
        full_bits = self._text_to_bits(message)
        capacity = (padded.shape[0]//self.bs) * (padded.shape[1]//self.bs)
        if len(full_bits) > capacity:
            raise ValueError(f"Message too large: {len(full_bits)} > {capacity} bits")

        stego = padded.copy()
        bit_idx = 0

        for i in range(0, padded.shape[0], self.bs):
            for j in range(0, padded.shape[1], self.bs):
                if bit_idx >= len(full_bits):
                    break
                block    = padded[i:i+self.bs, j:j+self.bs]
                dct_b    = dct(dct(block.T, norm='ortho').T, norm='ortho')
                q        = int(dct_b[self.ci, self.cj] / self.alpha)
                bit      = full_bits[bit_idx]
                # Force parity to match the bit
                if bit == '1' and q % 2 == 0: q += 1
                if bit == '0' and q % 2 != 0: q += 1
                dct_b[self.ci, self.cj] = q * self.alpha
                iblock = idct(idct(dct_b.T, norm='ortho').T, norm='ortho')
                stego[i:i+self.bs, j:j+self.bs] = iblock
                bit_idx += 1

        clipped = np.clip(stego[:h, :w], 0, 255).astype(np.uint8)
        return np.stack([clipped]*3, axis=-1) if image_array.ndim == 3 else clipped

    # ── extract (BLIND) ───────────────────────────────────────────────────────
    def extract(self, stego_array: np.ndarray) -> str:
        """No original image needed — reads parity of quantized coefficients."""
        gray = self._to_gray(stego_array)
        padded, _, _ = self._pad(gray)
        q = self.alpha

        all_bits = []
        msg_len  = None

        for i in range(0, padded.shape[0], self.bs):
            for j in range(0, padded.shape[1], self.bs):
                block = padded[i:i+self.bs, j:j+self.bs]
                dct_b = dct(dct(block.T, norm='ortho').T, norm='ortho')
                coef  = int(dct_b[self.ci, self.cj] / q)
                all_bits.append('1' if coef % 2 != 0 else '0')

                if msg_len is None and len(all_bits) >= 32:
                    try:
                        msg_len_candidate = ''.join(all_bits[:32])
                        if msg_len_candidate:
                            msg_len = int(msg_len_candidate, 2)
                            # Basic sanity check: message length should not be excessively large
                            if msg_len < 0 or msg_len > len(all_bits) * 8: # Arbitrary upper bound
                                return "" # Corrupted length
                        else:
                            return "" # No bits to form length prefix
                    except ValueError:
                        return "" # Corrupted length prefix

                if msg_len is not None and len(all_bits) >= 32 + msg_len:
                    return self._bits_to_text(''.join(all_bits))

        # If we reach here, it means the full message (as indicated by msg_len) was not extracted.
        # Attempt to return what we have, but it might be incomplete or corrupted.
        return self._bits_to_text(''.join(all_bits))

# ── Quick test ──
dct_stego = DCTStegoBlind(
    block_size  = CONFIG["dct_block_size"],
    coefficient = CONFIG["dct_coefficient"],
    alpha       = CONFIG["dct_alpha"]
)

_img0_dct = np.array(train_dataset[0][0].resize((512, 512)))
if _img0_dct.ndim == 2:
    _img0_dct = np.stack([_img0_dct]*3, axis=-1)

_sd   = dct_stego.embed(_img0_dct.copy(), _enc0)

try:
    _extracted_dct_raw = dct_stego.extract(_sd)
    _outd = encryptor.decrypt(_extracted_dct_raw)
    dct_round_trip_success = (_outd == structure_for_embedding(meta0))
except Exception as e:
    _outd = "DECRYPTION_FAILED"
    dct_round_trip_success = False
    print(f"   DCT (blind) decryption failed: {e}")
    # If you want to see the potentially corrupted extracted string for debugging:
    # print(f"   Corrupted extracted string (first 100 chars): {_extracted_dct_raw[:100]}...")

print(f" DCT (blind) round-trip: {dct_round_trip_success}")

cap_dct = (_img0_dct.shape[0]//8) * (_img0_dct.shape[1]//8)
print(f"   DCT capacity: {cap_dct} bits | Used: {len(_enc0)*8+32} bits")
# Output: DCT capacity: 4096 bits | Used: ~2700 bits

   DCT (blind) decryption failed: Nonce cannot be empty
 DCT (blind) round-trip: False
   DCT capacity: 4096 bits | Used: 2688 bits


## Step 8 — FFT Steganography (Frequency Domain — BLIND)
Embeds in mid-frequency band of the 2-D DFT magnitude spectrum.  
Same quantization-parity approach → blind extraction.

In [11]:
class FFTStego:
    """
    FFT steganography — magnitude quantization in mid-frequency band.
    Blind extraction (no original needed). Most robust to geometric attacks.
    """
    def __init__(self, freq_band=(50, 100), strength=5.0):
        self.lo, self.hi = freq_band
        self.q = strength

    def _positions(self, shape):
        h, w = shape
        cy, cx = h//2, w//2
        pts = []
        for dy in range(-self.hi, self.hi+1):
            for dx in range(-self.hi, self.hi+1):
                if self.lo <= (dy**2+dx**2)**0.5 <= self.hi:
                    y, x = cy+dy, cx+dx
                    if 0 <= y < h and 0 <= x < w:
                        pts.append((y, x))
        pts.sort()
        return pts

    def _text_to_bits(self, text):
        msg = ''.join(format(ord(c), '08b') for c in text)
        return format(len(msg), '032b') + msg

    def _bits_to_text(self, bits):
        n = int(bits[:32], 2)
        b = bits[32:32+n]
        return ''.join(chr(int(b[i:i+8], 2)) for i in range(0, len(b), 8))

    def _to_gray(self, arr):
        if arr.ndim == 3:
            return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float64)
        return arr.astype(np.float64)

    def embed(self, image_array: np.ndarray, message: str) -> np.ndarray:
        gray = self._to_gray(image_array)
        pts  = self._positions(gray.shape)
        full_bits = self._text_to_bits(message)
        if len(full_bits) > len(pts):
            raise ValueError(f"Message too large: {len(full_bits)} > {len(pts)}")

        F   = np.fft.fftshift(np.fft.fft2(gray))
        mag = np.abs(F)
        pha = np.angle(F)

        for idx, (y, x) in enumerate(pts[:len(full_bits)]):
            qv = int(mag[y, x] / self.q)
            if full_bits[idx] == '1' and qv % 2 == 0: qv += 1
            if full_bits[idx] == '0' and qv % 2 != 0: qv += 1
            mag[y, x] = qv * self.q

        stego_gray = np.clip(np.real(np.fft.ifft2(np.fft.ifftshift(mag * np.exp(1j*pha)))), 0, 255).astype(np.uint8)
        return np.stack([stego_gray]*3, axis=-1) if image_array.ndim == 3 else stego_gray

    def extract(self, stego_array: np.ndarray) -> str:
        gray = self._to_gray(stego_array)
        F    = np.fft.fftshift(np.fft.fft2(gray))
        mag  = np.abs(F)
        pts  = self._positions(gray.shape)
        bits = []
        for y, x in pts:
            qv = int(mag[y, x] / self.q)
            bits.append('1' if qv % 2 != 0 else '0')
            if len(bits) == 32:
                n = int(''.join(bits), 2)
                if n == 0: return ''
            if len(bits) > 32:
                needed = 32 + int(''.join(bits[:32]), 2)
                if len(bits) >= needed:
                    return self._bits_to_text(''.join(bits))
        return self._bits_to_text(''.join(bits))

# Quick test
fft_stego = FFTStego(
    freq_band = CONFIG["fft_freq_band"],
    strength  = CONFIG["fft_strength"]
)
_sf   = fft_stego.embed(_img0.copy(), _enc0)

try:
    _extracted_fft_raw = fft_stego.extract(_sf)
    _outf = encryptor.decrypt(_extracted_fft_raw)
    fft_round_trip_success = (_outf == structure_for_embedding(meta0))
except Exception as e:
    _outf = "DECRYPTION_FAILED"
    fft_round_trip_success = False
    print(f"   FFT (blind) decryption failed: {e}")
    # If you want to see the potentially corrupted extracted string for debugging:
    # print(f"   Corrupted extracted string (first 100 chars): {_extracted_fft_raw[:100]}...")

print(f" FFT (blind) round-trip: {fft_round_trip_success}")
cap_fft = len(fft_stego._positions(_img0[:,:,0].shape))
print(f"   FFT capacity: {cap_fft} bits | Used: {len(_enc0)*8+32} bits")

   FFT (blind) decryption failed: string argument should contain only ASCII characters
 FFT (blind) round-trip: False
   FFT capacity: 23592 bits | Used: 2688 bits


## Step 9 — Quality Metrics: PSNR, MSE, SSIM

In [12]:
def calculate_metrics(original: np.ndarray, stego: np.ndarray) -> dict:
    """Returns PSNR (dB), MSE, and SSIM for a stego-image pair."""
    o = original.astype(np.float64)
    s = stego.astype(np.float64)

    mse  = float(np.mean((o - s) ** 2))
    psnr = float(20 * np.log10(255.0 / np.sqrt(mse))) if mse > 0 else float('inf')

    og = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY) if original.ndim==3 else original
    sg = cv2.cvtColor(stego,    cv2.COLOR_RGB2GRAY) if stego.ndim==3    else stego
    ssim_v = float(ssim_func(og, sg, data_range=255))

    return {'psnr': round(psnr,4), 'mse': round(mse,6), 'ssim': round(ssim_v,6)}

def quality_label(psnr):
    if psnr >= 45: return "Excellent (imperceptible)"
    if psnr >= 35: return "Good (barely noticeable)"
    if psnr >= 25: return "Fair (visible distortion)"
    return "  Poor"

# ── Demo on sample ────────────────────────────────────────────────────────────
print(" Quality Metrics — sample image[0]")
print(f"{'Method':<8} {'PSNR (dB)':>10} {'MSE':>10} {'SSIM':>8} Rating")
print("-" * 65)
for name, original_img, stego_img in [("LSB", _img0, _s), ("DCT", _img0_dct, _sd), ("FFT", _img0, _sf)]:
    m = calculate_metrics(original_img, stego_img)
    print(f"{name:<8} {m['psnr']:>10.4f} {m['mse']:>10.6f} {m['ssim']:>8.6f}  {quality_label(m['psnr'])}")

 Quality Metrics — sample image[0]
Method    PSNR (dB)        MSE     SSIM Rating
-----------------------------------------------------------------
LSB         68.6554   0.008862 0.999973  Excellent (imperceptible)
DCT         50.1567   0.627213 0.995128  Excellent (imperceptible)
FFT         51.1806   0.495476 0.998048  Excellent (imperceptible)


## Step 10 — Robustness Testing (7 attack types)

In [13]:
def _add_salt_pepper(img, prob=0.01):
    out = img.copy()
    mask = np.random.random(img.shape[:2])
    out[mask < prob/2]   = 0
    out[mask > 1-prob/2] = 255
    return out.astype(np.uint8)

def _jpeg_compress(img, quality=75):
    _, enc = cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                          [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)

ATTACKS = {
    "Gaussian Noise σ=5"  : lambda i: np.clip(i.astype(np.float32) + np.random.normal(0, 5, i.shape), 0, 255).astype(np.uint8),
    "Salt & Pepper 1%"    : lambda i: _add_salt_pepper(i, 0.01),
    "JPEG Q=75"           : lambda i: _jpeg_compress(i, 75),
    "JPEG Q=50"           : lambda i: _jpeg_compress(i, 50),
    "Resize 0.5× → 2×"   : lambda i: cv2.resize(cv2.resize(i,(i.shape[1]//2,i.shape[0]//2)), (i.shape[1],i.shape[0])),
    "Gaussian Blur 3×3"   : lambda i: cv2.GaussianBlur(i, (3,3), 0),
    "Median Filter 3×3"   : lambda i: cv2.medianBlur(i, 3),
}

def robustness_test(stego_img, extractor_fn, original_message):
    """Test extraction after each attack. Returns dict of results."""
    results = {}
    for attack_name, attack_fn in ATTACKS.items():
        try:
            attacked_img  = attack_fn(stego_img.copy())
            extracted_enc = extractor_fn(attacked_img)
            decrypted_msg = encryptor.decrypt(extracted_enc)
            exact  = decrypted_msg == original_message
            char_a = sum(a==b for a,b in zip(decrypted_msg, original_message)) / max(len(original_message),1)
        except Exception as e:
            exact, char_a = False, 0.0
        results[attack_name] = {'exact_match': exact, 'char_accuracy': round(char_a, 3)}
    return results

print("  Robustness test — LSB on sample image:")
print(f"{'Attack':<25} {'Exact?':>6} {'Char Acc':>10}")
print("-" * 45)
_orig_msg = structure_for_embedding(meta0)
for atk, res in robustness_test(_s, lsb_stego.extract, _orig_msg).items():
    tick = "✅" if res['exact_match'] else "❌"
    print(f"{atk:<25} {tick:>6} {res['char_accuracy']:>10.3f}")

  Robustness test — LSB on sample image:
Attack                    Exact?   Char Acc
---------------------------------------------
Gaussian Noise σ=5             ❌      0.000
Salt & Pepper 1%               ❌      0.000
JPEG Q=75                      ❌      0.000
JPEG Q=50                      ❌      0.000
Resize 0.5× → 2×               ❌      0.000
Gaussian Blur 3×3              ❌      0.000
Median Filter 3×3              ❌      0.000


## Step 11 — Evaluation on Multiple Images (Train / Val / Test)
Runs the full pipeline across `eval_n_images` images from all three splits.

In [14]:
## Step 11 — LSB Evaluation (Train / Val / Test)

# LSB-only: encrypt → embed → extract → decrypt
# DCT and FFT are skipped until those extractors are fixed.

encryptor = MediGuardEncryptor(ENCRYPTION_PASSWORD)
lsb_stego = LSBStego(bits=CONFIG["lsb_bits"])

all_rows = []
for ds, name in [(train_dataset, "train"), (val_dataset, "val"), (test_dataset, "test")]:
    all_rows += evaluate_lsb_split(
        ds,
        name,
        encryptor,
        lsb_stego,
        n=CONFIG["eval_n_images"],
        image_size=CONFIG["image_size"],
    )

df_eval = pd.DataFrame(all_rows)

print("\n" + "=" * 70)
print("LSB SUMMARY — embed → extract → decrypt")
print("=" * 70)
for split in ["train", "val", "test"]:
    sub = df_eval[df_eval["split"] == split]
    if sub.empty:
        continue
    psnr_mean = sub["LSB_psnr"].mean()
    ssim_mean = sub["LSB_ssim"].mean()
    acc_mean  = sub["LSB_success"].mean()
    psnr_str  = f"{psnr_mean:.2f} dB" if not pd.isna(psnr_mean) else "N/A"
    ssim_str  = f"{ssim_mean:.6f}"    if not pd.isna(ssim_mean) else "N/A"
    print(f"  {split:<6} | PSNR: {psnr_str} | SSIM: {ssim_str} | Accuracy: {acc_mean * 100:.1f}%")



  Evaluating train (20 images) ...
    5/20 done
    10/20 done
    15/20 done
    20/20 done

  Evaluating val (20 images) ...
    5/20 done
    10/20 done
    15/20 done
    20/20 done

  Evaluating test (20 images) ...
    5/20 done
    10/20 done
    15/20 done
    20/20 done

LSB SUMMARY — embed → extract → decrypt
  train  | PSNR: 71.89 dB | SSIM: 0.999992 | Accuracy: 100.0%
  val    | PSNR: 71.94 dB | SSIM: 0.999994 | Accuracy: 100.0%
  test   | PSNR: 71.89 dB | SSIM: 0.999992 | Accuracy: 100.0%


## Step 12 — Save All Artifacts

In [ ]:
# ── Save LSB artifacts to mediguard_outputs/ ───────────────────────────────

csv_path = METRICS_DIR / "evaluation_results.csv"
df_eval.to_csv(csv_path, index=False)
print(f" Metrics CSV     → {csv_path}")

summary = build_lsb_summary(df_eval, CONFIG)
json_path = METRICS_DIR / "summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f" Summary JSON    → {json_path}")

# One LSB stego PNG per split
for ds, split_name in [(train_dataset, "train"), (val_dataset, "val"), (test_dataset, "test")]:
    pil_img, _ = ds[0]
    img_arr = pil_to_rgb_array(pil_img, CONFIG["image_size"])
    meta = get_patient_metadata(0, dataset=ds, use_ai=False)
    message = structure_for_embedding(meta)
    result = lsb_round_trip(encryptor, lsb_stego, img_arr, message)
    p = STEGO_DIR / f"stego_LSB_{split_name}.png"
    Image.fromarray(result["stego_image"]).save(p)
    print(f" Stego PNG       → {p}  (PSNR {result['metrics']['psnr']} dB)")

# Sync frontend dashboard
export_frontend_metrics(summary, PROJECT_ROOT / "frontend" / "public" / "data" / "metrics.json")

if AI_AVAILABLE:
    model_path = MODELS_DIR / "densenet121_nih_xray.pth"
    torch.save(ai_model.state_dict(), model_path)
    print(f" AI model        → {model_path}")

print(f"\n All LSB artifacts saved under: {OUTPUT_DIR}")


## Step 13 — Visualisation (LSB only — DCT/FFT not in df_eval)

In [ ]:
## Step 13 — Visualisation (LSB)

# ── Prepare images ────────────────────────────────────────────────────────────
pil_img0, _ = train_dataset[0]

img0_224 = np.array(pil_img0.resize((224, 224)))
if img0_224.ndim == 2: img0_224 = np.stack([img0_224] * 3, axis=-1)

meta0_vis = get_patient_metadata(0, use_ai=False)
msg0      = structure_for_embedding(meta0_vis)
enc0      = encryptor.encrypt(msg0)

stego_lsb = lsb_stego.embed(img0_224.copy(), enc0)
m_lsb = calculate_metrics(img0_224, stego_lsb)

# ── LSB comparison figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
fig.suptitle("SpectraVault — LSB Steganography (working method)", fontsize=14, fontweight='bold', y=1.01)

imgs   = [img0_224, stego_lsb]
titles = [
    'Original X-ray',
    f'LSB  PSNR={m_lsb["psnr"]:.1f} dB\nSSIM={m_lsb["ssim"]:.4f}',
]
colors = ['#888888', '#2196F3']

for col, (img, title) in enumerate(zip(imgs, titles)):
    axes[0, col].imshow(img, cmap='gray' if img.ndim == 2 else None)
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].axis('off')
    axes[1, col].hist(img.ravel(), bins=256, range=(0, 256),
                      color=colors[col], alpha=0.8, density=True)
    axes[1, col].set_title('Histogram', fontsize=9)
    axes[1, col].set_xlim(0, 255)

for col in range(2, axes.shape[1]):
    axes[0, col].axis('off')
    axes[1, col].axis('off')

plt.tight_layout()
fig_path = FIGS_DIR / "comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Figure -> {fig_path}")

# ── Bar chart — LSB metrics across splits (from df_eval) ─────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle("LSB Evaluation — Train / Val / Test", fontsize=13, fontweight='bold')

plot_specs = [
    ('LSB_psnr', 'PSNR (dB)'),
    ('LSB_ssim', 'SSIM'),
    ('LSB_success', 'Extraction Accuracy'),
]
splits = ['train', 'val', 'test']

for ax, (col_name, ylabel) in zip(axes2, plot_specs):
    if col_name not in df_eval.columns:
        ax.set_title(f'{ylabel} (no data)')
        continue
    vals = [df_eval[df_eval['split'] == s][col_name].mean() for s in splits]
    ax.bar(splits, vals, color='#2196F3', alpha=0.85)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)

plt.tight_layout()
fig2_path = FIGS_DIR / "split_comparison.png"
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Figure -> {fig2_path}")

## Step 14 — Download outputs (Colab only; skip on local PC)

In [ ]:
import shutil
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    zip_path = "/content/mediguard_outputs.zip"
    shutil.make_archive("/content/mediguard_outputs", "zip", OUTPUT_DIR)
    print(f" Archive: {zip_path}")
    files.download(zip_path)
else:
    print("Running locally — skip Colab download.")
    print(f"Your outputs are already here: {OUTPUT_DIR.resolve()}")
    print("Open that folder in File Explorer to view CSV, JSON, and stego PNGs.")